# NB13 — Business Error Diagnosis & Action Plan

## Objective

NB13 extends the business-facing evaluation introduced in NB12.

NB12 showed that NB11 is operationally safer than NB9 in the most costly forecast segments.

NB13 focuses on the next business question:

Where exactly are the most expensive forecast errors concentrated,
and what actions would reduce operational risk further?

This notebook is not a model training notebook.

Its purpose is to identify:

- where forecast errors are most expensive
- which store/family combinations drive operational risk
- where NB11 improves over NB9
- where NB11 still fails
- what actions are most likely to reduce business risk

NB13 is a diagnostic and action-planning notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))

In [2]:
# Setup
ARTIFACTS_DIR = Path("../artifacts/nb13_business_error_diagnosis")
NB9_DIR = Path("../artifacts/nb9_autogluon_tabular")
NB11_DIR = Path("../artifacts/nb11_autogluon_extended_budget")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

NB9_PRED_PATH = NB9_DIR / "nb9_fold_predictions.csv"
NB11_PRED_PATH = NB11_DIR / "nb11_fold_predictions.csv"

print("Setup OK")

Setup OK


## Load Prediction Artifacts

In [3]:
nb9 = pd.read_csv(NB9_PRED_PATH, parse_dates=["date"])
nb11 = pd.read_csv(NB11_PRED_PATH, parse_dates=["date"])

## Standardize Frames

In [4]:
def prepare_eval_frame(df, model_name):
    out = df.copy()
    out["model"] = model_name
    out["error"] = out["sales"] - out["prediction"]
    out["abs_error"] = np.abs(out["error"])
    out["weighted_abs_error"] = out["abs_error"] * out["sales"]
    return out

nb9_eval = prepare_eval_frame(nb9, "NB9")
nb11_eval = prepare_eval_frame(nb11, "NB11")

## Build Paired Comparison Frame

To compare both models row-by-row, forecasts are aligned on the same observations.

In [5]:
pair_df = nb9_eval.merge(
    nb11_eval,
    on=["date", "store_nbr", "family", "sales", "onpromotion", "fold"],
    suffixes=("_nb9", "_nb11")
)

pair_df["mae_gain"] = pair_df["abs_error_nb9"] - pair_df["abs_error_nb11"]
pair_df["weighted_gain"] = pair_df["weighted_abs_error_nb9"] - pair_df["weighted_abs_error_nb11"]

print("Paired shape:", pair_df.shape)

Paired shape: (49896, 18)


## Worst Store-Family Risk Pockets

Identify store/family combinations generating the highest operational error.

In [6]:
risk_pockets = (
    pair_df.groupby(["store_nbr", "family"])
    .agg(
        nb9_weighted_error=("weighted_abs_error_nb9", "sum"),
        nb11_weighted_error=("weighted_abs_error_nb11", "sum"),
        weighted_gain=("weighted_gain", "sum"),
        avg_sales=("sales", "mean"),
        promo_days=("onpromotion", lambda x: (x > 0).sum()),
    )
    .reset_index()
    .sort_values("nb11_weighted_error", ascending=False)
)

display(risk_pockets.head(20))

,store_nbr,family,nb9_weighted_error,nb11_weighted_error,weighted_gain,avg_sales,promo_days
1422,44,3,3.262572e+08,3.048351e+08,2.142205e+07,9973.607143,28
1464,45,12,4.119691e+08,2.701432e+08,1.418259e+08,11195.821429,28
1521,47,3,2.887740e+08,2.294150e+08,5.935904e+07,9088.821429,28
1497,46,12,3.017904e+08,2.279119e+08,7.387848e+07,9761.071429,28
1530,47,12,2.766669e+08,2.139560e+08,6.271091e+07,10403.428571,28
903,28,12,1.949339e+08,2.059364e+08,-1.100255e+07,5202.214500,28
1431,44,12,1.863896e+08,1.979719e+08,-1.158229e+07,9948.857143,28
1455,45,3,2.178038e+08,1.886474e+08,2.915632e+07,9342.285714,28
1290,40,3,1.728026e+08,1.723082e+08,4.944352e+05,4933.500000,28
1752,54,3,1.572167e+08,1.659087e+08,-8.692035e+06,3717.678571,28


## Worst Promo Failures

In [7]:
worst_promo = (
    pair_df[pair_df["onpromotion"] > 0]
    .sort_values("weighted_abs_error_nb11", ascending=False)
    [[
        "date", "store_nbr", "family", "sales",
        "prediction_nb9", "prediction_nb11",
        "weighted_abs_error_nb9", "weighted_abs_error_nb11"
    ]]
)

display(worst_promo.head(20))

,date,store_nbr,family,sales,prediction_nb9,prediction_nb11,weighted_abs_error_nb9,weighted_abs_error_nb11
39827,2017-07-30,44,3,18340.000,14135.1580,15089.82800,7.711680e+07,5.960815e+07
25308,2017-08-12,28,12,9841.523,5398.0650,5196.08900,4.373039e+07,4.571815e+07
36396,2017-08-12,40,12,10739.000,5717.2256,6624.07500,5.392884e+07,4.419018e+07
41003,2017-07-30,45,12,16847.000,13909.7690,14351.20000,4.948353e+07,4.204674e+07
25309,2017-08-13,28,12,11002.664,7654.0040,7253.14450,3.684418e+07,4.125470e+07
42605,2017-08-05,47,3,14469.000,10270.3840,11716.79900,6.074977e+07,3.982160e+07
42599,2017-07-30,47,3,15874.000,12714.0400,13448.70000,5.016121e+07,3.849921e+07
39826,2017-07-29,44,3,16087.000,13581.9030,13884.03100,4.029950e+07,3.543916e+07
36136,2017-08-04,40,3,8272.000,4489.1323,4196.20700,3.129188e+07,3.371496e+07
49074,2017-08-06,54,3,8552.000,3906.6394,4615.33350,3.972712e+07,3.366637e+07


## Worst Underforecast Cases

In [8]:
pair_df["underforecast_nb11"] = (pair_df["prediction_nb11"] < pair_df["sales"]).astype(int)

worst_underforecast = (
    pair_df[pair_df["underforecast_nb11"] == 1]
    .sort_values("weighted_abs_error_nb11", ascending=False)
    [[
        "date", "store_nbr", "family", "sales",
        "prediction_nb11", "weighted_abs_error_nb11"
    ]]
)

display(worst_underforecast.head(20))

,date,store_nbr,family,sales,prediction_nb11,weighted_abs_error_nb11
39827,2017-07-30,44,3,18340.000,15089.82800,5.960815e+07
25308,2017-08-12,28,12,9841.523,5196.08900,4.571815e+07
36396,2017-08-12,40,12,10739.000,6624.07500,4.419018e+07
41003,2017-07-30,45,12,16847.000,14351.20000,4.204674e+07
25309,2017-08-13,28,12,11002.664,7253.14450,4.125470e+07
42605,2017-08-05,47,3,14469.000,11716.79900,3.982160e+07
42599,2017-07-30,47,3,15874.000,13448.70000,3.849921e+07
39826,2017-07-29,44,3,16087.000,13884.03100,3.543916e+07
36136,2017-08-04,40,3,8272.000,4196.20700,3.371496e+07
49074,2017-08-06,54,3,8552.000,4615.33350,3.366637e+07


## Where NB11 Improves Most

In [9]:
best_nb11_gains = (
    pair_df.sort_values("weighted_gain", ascending=False)
    [[
        "date", "store_nbr", "family", "sales",
        "prediction_nb9", "prediction_nb11",
        "weighted_gain"
    ]]
)

display(best_nb11_gains.head(20))

,date,store_nbr,family,sales,prediction_nb9,prediction_nb11,weighted_gain
41010,2017-08-06,45,12,15190.0,12462.8880,14634.8290,3.299178e+07
43782,2017-08-06,48,12,14062.0,11685.6150,14100.8530,3.287037e+07
42592,2017-07-23,47,3,14933.0,12178.9680,14136.8170,2.923656e+07
40744,2017-07-23,45,3,14755.0,12902.6970,14622.9260,2.538198e+07
40996,2017-07-23,45,12,15686.0,12630.6190,14038.7550,2.208802e+07
42605,2017-08-05,47,3,14469.0,10270.3840,11716.7990,2.092818e+07
41933,2017-08-05,46,12,13244.0,11554.2170,13413.3660,2.013640e+07
41004,2017-07-31,45,12,11544.0,9732.4960,11632.3120,1.989253e+07
42852,2017-07-31,47,12,12238.0,9379.4060,10932.2650,1.900389e+07
41009,2017-08-05,45,12,14189.0,12751.5060,14326.2710,1.844886e+07


## Where NB11 Still Fails

In [10]:
nb11_failures = (
    pair_df.sort_values("weighted_abs_error_nb11", ascending=False)
    [[
        "date", "store_nbr", "family", "sales",
        "prediction_nb11", "weighted_abs_error_nb11"
    ]]
)

display(nb11_failures.head(20))

,date,store_nbr,family,sales,prediction_nb11,weighted_abs_error_nb11
39827,2017-07-30,44,3,18340.000,15089.82800,5.960815e+07
25308,2017-08-12,28,12,9841.523,5196.08900,4.571815e+07
36396,2017-08-12,40,12,10739.000,6624.07500,4.419018e+07
41003,2017-07-30,45,12,16847.000,14351.20000,4.204674e+07
25309,2017-08-13,28,12,11002.664,7253.14450,4.125470e+07
42605,2017-08-05,47,3,14469.000,11716.79900,3.982160e+07
42599,2017-07-30,47,3,15874.000,13448.70000,3.849921e+07
39826,2017-07-29,44,3,16087.000,13884.03100,3.543916e+07
36136,2017-08-04,40,3,8272.000,4196.20700,3.371496e+07
49074,2017-08-06,54,3,8552.000,4615.33350,3.366637e+07


## Action Layer

Translate diagnostic findings into operational actions.

In [11]:
action_table = pd.DataFrame([
    {
        "issue": "Promo spikes",
        "diagnosis": "Largest forecast errors remain concentrated in promotional periods",
        "recommended_action": "Add stronger promo uplift and promo-history features"
    },
    {
        "issue": "High-volume underforecast",
        "diagnosis": "Largest costly misses still occur in top-selling high-volume segments",
        "recommended_action": "Apply asymmetric underforecast penalty or bias correction"
    },
    {
        "issue": "Store-family concentration",
        "diagnosis": "Operational risk is concentrated in a limited set of store/family pockets",
        "recommended_action": "Consider segment-specific modeling or local overrides"
    },
    {
        "issue": "Residual high-risk slices",
        "diagnosis": "NB11 improves overall business risk but still fails on a few extreme slices",
        "recommended_action": "Introduce forecast confidence and alerting layer"
    }
])

display(action_table)

,issue,diagnosis,recommended_action
0,Promo spikes,Largest forecast errors remain concentrated in...,Add stronger promo uplift and promo-history fe...
1,High-volume underforecast,Largest costly misses still occur in top-selli...,Apply asymmetric underforecast penalty or bias...
2,Store-family concentration,Operational risk is concentrated in a limited ...,Consider segment-specific modeling or local ov...
3,Residual high-risk slices,NB11 improves overall business risk but still ...,Introduce forecast confidence and alerting layer


## Save Artifacts

In [12]:
risk_pockets.to_csv(ARTIFACTS_DIR / "nb13_risk_pockets.csv", index=False)
worst_promo.to_csv(ARTIFACTS_DIR / "nb13_worst_promo.csv", index=False)
worst_underforecast.to_csv(ARTIFACTS_DIR / "nb13_worst_underforecast.csv", index=False)
best_nb11_gains.to_csv(ARTIFACTS_DIR / "nb13_best_nb11_gains.csv", index=False)
nb11_failures.to_csv(ARTIFACTS_DIR / "nb13_nb11_failures.csv", index=False)
action_table.to_csv(ARTIFACTS_DIR / "nb13_action_table.csv", index=False)

print("Saved NB13 artifacts to:", ARTIFACTS_DIR.resolve())

Saved NB13 artifacts to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb13_business_error_diagnosis


## Final Conclusion

NB13 identifies where forecast risk is most concentrated
and translates model behavior into operational actions.

NB12 showed which model is safer.

NB13 shows where to intervene next.

This notebook closes the loop between:

- model performance
- operational risk
- business action